In [ ]:
import stixpy
from stixpy.product import Product
import numpy as np
import astropy
from astropy.io import fits
from astropy import units as u
# from stixdcpy.net import Request as jreq
from stixpy.coordinates.flare_location import stx_estimate_flare_location
from stixpy.calibration.flare_location import estimate_flare_location
from astropy.io import fits

from sunkit_spex.fitting.fitter import Fitter
from sunkit_spex.models.physical.thermal import ThermalEmission
from sunkit_spex.models.physical.nonthermal import ThickTarget
from sunkit_spex.models.physical.albedo import Albedo
from sunkit_spex.models.instrument_response import MatrixModel
from sunkit_spex.models.scaling import InverseSquareFluxScaling
import numpy as np
import astropy.units as u
import numpy as np
from ndcube import NDMeta
from ndcube.extra_coords import QuantityTableCoordinate, TimeTableCoordinate
import astropy.units as u
from astropy.coordinates import SpectralCoord
from astropy.time import Time
# from sunkit_spex.spectrum import Spectrum
from sunkit_spex.spectrum.spectrum import Spectrum, SpectralAxis
# from astropy.nddata import NDData, StdDevUncertainty
from sunkit_spex.spectrum.uncertainty import PoissonUncertainty

In [ ]:
import os
import urllib.request
if not os.path.exists('./data'): os.mkdir('./data/')
if not os.path.exists('./data/solo_L1_stix-sci-xray-cpd_20240308T193915-20240308T203235_V02_2403087339-57240.fits'):
    urllib.request.urlretrieve('https://cloud.aip.de/index.php/s/E9MYCsHSijbqtNz/download/solo_L1_stix-sci-xray-cpd_20240308T193915-20240308T203235_V02_2403087339-57240.fits',
                               './data/solo_L1_stix-sci-xray-cpd_20240308T193915-20240308T203235_V02_2403087339-57240.fits')
if not os.path.exists('./data/solo_L1_stix-sci-xray-cpd_20240310T115906-20240310T121540_V02_2403109216-57290.fits'):
    urllib.request.urlretrieve('https://cloud.aip.de/index.php/s/jxya5RixMFsZd4M/download/solo_L1_stix-sci-xray-cpd_20240310T115906-20240310T121540_V02_2403109216-57290.fits',
                               './data/solo_L1_stix-sci-xray-cpd_20240310T115906-20240310T121540_V02_2403109216-57290.fits')

data_dir = './data/'
file = 'solo_L1_stix-sci-xray-cpd_20240310T115906-20240310T121540_V02_2403109216-57290.fits'
file_bkg = 'solo_L1_stix-sci-xray-cpd_20240308T193915-20240308T203235_V02_2403087339-57240.fits'


In [ ]:
spec_prod = Product(data_dir+file)
spec_prod.plot_timeseries()

In [ ]:
# t_range = [spec_prod.meta['DATE-BEG'],Time(spec_prod.meta['DATE-END'])-15*u.min]
# t_cent =  spec_prod.meta['DATE-AVG']

# flare_location= estimate_flare_location(data_dir+file, t_range)

In [ ]:
# flare_location_hpc = flare_location['hpc']
# flare_location_stx = flare_location['stx']


# print(flare_location_hpc)
# print(flare_location_stx)

In [ ]:
t_range = [spec_prod.meta['DATE-BEG'],Time(spec_prod.meta['DATE-END'])-15*u.min]
t_cent =  spec_prod.meta['DATE-AVG']

flare_location= stx_estimate_flare_location(data_dir+file, t_range)

flare_location_hpc = flare_location['hpc']
flare_location_stx = flare_location['stx']


print(flare_location_hpc)
print(flare_location_stx)

In [ ]:
# flare_aux=jreq.request_flare_light_time_and_angle(utc=t_cent, 
#                                                   flare_x=1342.04276689, 
#                                                   flare_y=flare_location_hpc[1], 
#                                                   observer='stix')

# flare_angle = flare_aux['flare_norm_solo_deg']


# flare_angle

In [ ]:
bkg = Product(data_dir+file_bkg)

In [ ]:
srm_dat = spec_prod.get_masked_srm(flare_location=flare_location)

In [ ]:
# t_range = ["2024-03-10T12:00:55","2024-03-10T12:40:00"]
t_range = ["2024-03-10T12:05:55","2024-03-10T12:06:00"]

spectrum = spec_prod.get_spec_obj(event_time_range=t_range,
                             srm_dictionary=srm_dat,
                             bkg_data=bkg,
                             flare_location=flare_location)

In [ ]:
model = (((ThermalEmission() + ThickTarget()) | Albedo()) * InverseSquareFluxScaling() ) | MatrixModel()

model.param_names

In [ ]:
hasattr(model,'_leaflist')

In [ ]:
th = ThermalEmission()
th.verify_dims_in_fitting


In [ ]:
# if isinstance(model, CompoundModel):
#     for component in model._leaflist:
#         if hasattr(component, 'verify_dims_in_fitting'):
#             print(f"{component.name}: {component.verify_dims_in_fitting}")

In [ ]:
model.left.left.left.left.verify_dims_in_fitting

In [ ]:
model.break_energy_1 =1500*u.keV
model.break_energy_1.fixed = True
model.total_eflux_1.fixed = False
model.low_e_cutoff_1.min=8*u.keV
model.low_e_cutoff_1.max=60*u.keV

In [ ]:
fitter = Fitter(model=model,
                spectrum_object=spectrum)

In [ ]:
fitter.fit_range = [4.5*u.keV, 85*u.keV]

In [ ]:
fitter._spectrum_object.__dict__

In [ ]:
fitter._spectrum_object.meta['distance']
fitter._spectrum_object.meta['angle']

In [ ]:
fitter.model.fixed

In [ ]:
fitter.do_fit()

In [ ]:
print(fitter.fitted_model)

In [ ]:
fitter.plot_fit_results()